In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\Lenovo\Projects\Portfolio\DataCo Smart Supply Chain\archive\DataCoSupplyChainDataset.csv", encoding="latin1")

In [3]:
print("Initial shape:", df.shape)

# --- Drop PII and unneeded columns ---
cols_to_drop = [
    "Customer Email", "Customer Fname", "Customer Lname", "Customer Password",
    "Customer Street",          # street-level PII, city/state/country is enough
    "Product Image",            # image URL, not useful for KPI analysis
    "Product Description",      # long text, not needed for dashboard
    "Product Status",           # mostly constant/uninformative in this dataset — verify before assuming
        'Order Customer Id',       # simpan: Customer Id
    'Product Category Id',     # simpan: Category Id
    'Order Item Cardprod Id',  # simpan: Product Card Id
    'Product Price',           # simpan: Order Item Product Price
    'Benefit per order',       # simpan: Order Profit Per Order
    'Sales per customer',      # simpan: Order Item Total
    'Late_delivery_risk',      # simpan: is_late
    'Order Zipcode', 'Customer Zipcode', 'Customer City',
    'Customer State', 'Customer Country', 'Latitude', 'Longitude',
    'Category Id', 'Department Id', 'Product Card Id',
    'Order Item Discount Rate', 'Order Item Profit Ratio', 'Sales'
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

Initial shape: (180519, 53)


In [4]:
df = df.rename(columns={"Type":"Payment Type"})

In [47]:
df.duplicated().any()

False

In [48]:
df.isnull().sum()

Payment Type                     0
Days for shipping (real)         0
Days for shipment (scheduled)    0
Delivery Status                  0
Category Name                    0
Customer Id                      0
Customer Segment                 0
Department Name                  0
Market                           0
Order City                       0
Order Country                    0
order date (DateOrders)          0
Order Id                         0
Order Item Discount              0
Order Item Id                    0
Order Item Product Price         0
Order Item Quantity              0
Order Item Total                 0
Order Profit Per Order           0
Order Region                     0
Order State                      0
Order Status                     0
Product Name                     0
shipping date (DateOrders)       0
Shipping Mode                    0
dtype: int64

In [5]:
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])
df['shipping date (DateOrders)'] = pd.to_datetime (df['shipping date (DateOrders)'])
df.columns = (df.columns.str.lower().str.replace(r'\s*\(.*\)', '', regex=True).str.strip().str.replace(' ','_'))

In [6]:
df['shipping_delay_days'] = df['days_for_shipping']-df['days_for_shipment']
df['is_late'] = (df['shipping_delay_days']>0).astype(int)

In [7]:
for col in ['shipping_mode', 'delivery_status', 'order_status', 'market', 'customer_segment']:
    print(df[col].value_counts(), '\n')

df.describe()
(df['shipping_date'] >= df['order_date']).mean()   # harusnya 1.0

shipping_mode
Standard Class    107752
Second Class       35216
First Class        27814
Same Day            9737
Name: count, dtype: int64 

delivery_status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64 

order_status
COMPLETE           59491
PENDING_PAYMENT    39832
PROCESSING         21902
PENDING            20227
CLOSED             19616
ON_HOLD             9804
SUSPECTED_FRAUD     4062
CANCELED            3692
PAYMENT_REVIEW      1893
Name: count, dtype: int64 

market
LATAM           51594
Europe          50252
Pacific Asia    41260
USCA            25799
Africa          11614
Name: count, dtype: int64 

customer_segment
Consumer       93504
Corporate      54789
Home Office    32226
Name: count, dtype: int64 



1.0

In [13]:
df['is_late'].value_counts(normalize=True)

is_late
1    0.572793
0    0.427207
Name: proportion, dtype: float64